# Import bibliotek

In [1]:
import autoroot  # noqa
import joblib
from dotenv import load_dotenv
from rich import print as pprint
from grawiki.db.falkordb import FalkorGraphDB
from grawiki.rag.graph_rag import GraphRAG
from pathlib import Path
from tqdm.auto import tqdm

/home/filip/projects/grawiki/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

True

# RAG init

In [3]:
database = FalkorGraphDB(db_path="local_falcor.db", graph_name="grawiki")

In [4]:
rag = GraphRAG(
    db=database,
    model="openrouter:openai/gpt-5-mini",
    embedding_model="openrouter:openai/text-embedding-3-small",
    max_workers=10,
    resolve_entities_on_ingest=True,
    entity_resolution_threshold=0.9,
)

# Data reading / loading

In [5]:
docs = list(Path("experimental_data").glob("*.txt"))

In [6]:
for doc in tqdm(docs, desc="Ingesting documents"):
    await rag.ingest(doc)

Ingesting documents: 100%|██████████| 3/3 [11:42<00:00, 234.30s/it]


# Deduplication

In [7]:
report = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.9, dry_run=True
)

# Debug

In [12]:
res = await rag.search("Knowledge graph?", limit=5)

In [ ]:
rag.remember()

In [49]:
from pydantic_ai import Agent
from dataclasses import asdict

agent = Agent(
    model="openrouter:openai/gpt-5-mini",
    name="Knowledge assistant",
    system_prompt="""You are a helpful assistant that answers user queries based on the knowledge base at your disposal.
    When answering questions - use knowledge graph search tool. You can also create memories of past conversations 
    and recall past memories if needed.
    Create memories when you acquire imporant information about the user during the conversation that might be useful in the future. You can use the search tool to find relevant information in the knowledge graph.""",
)


@agent.tool_plain
async def search_knowledge_graph(query: str, limit: int = 5) -> dict:
    """Search the knowledge graph for relevant information. Accepts query and limit for number of results. Returns a formatted string of results. Returns dict of nested nodes"""
    results = await rag.search(query, limit=limit)
    results_as_dict = []
    for hit in results:
        node_dict = asdict(hit)
        node_dict["node"] = hit.node.model_dump()
        results_as_dict.append(node_dict)
    return results_as_dict


@agent.tool_plain
async def create_memory(
    memory_content: str, memory_name: str, related_node_ids: list[int]
) -> str:
    """Create a memory of past conversations. Accepts memory content, memory name and related node ids. Returns a confirmation message."""
    await rag.remember(
        memory=memory_content, name=memory_name, related_node_ids=related_node_ids
    )
    return f"Memory '{memory_name}' created successfully."

In [45]:
result = await agent.run("What is a knowledge graph and what do you know about it?")

In [46]:
pprint(result.output)

Short definition
- A knowledge graph (KG) is a structured, graph‑based representation of knowledge: nodes (entities or concepts) 
connected by typed edges (relationships), often with properties on nodes and edges and an explicit schema or 
ontology describing allowed classes and relations. Its basic unit is usually a triple: (subject — relation — 
object).

What it contains (main components)
- Entities / nodes: people, places, products, documents, concepts, etc.  
- Relations / edges: typed links like “authorOf”, “locatedIn”, “acquiredBy”.  
- Properties: attributes on nodes/edges (e.g., name, date).  
- Schema / ontology: types and constraints that define classes and relation types.  
- Triples: subject-relation-object facts (e.g., (Alice — wrote — Paper1)).

Common data models & query languages
- RDF / triples + SPARQL: a W3C standard for semantic data and querying.  
- Property graph (used by Neo4j, etc.): nodes/edges with arbitrary properties; queried with Cypher or graph DB 
APIs.

How KGs are built
- From structured sources (DBpedia, Wikidata, databases).  
- From unstructured text via information extraction (NER, relation extraction) or LLM-assisted extraction.  
- By manual curation or hybrid pipelines (automated extraction + human validation).

How KGs are used & processed
- Classic graph analytics: path queries, centrality, pattern matching, clustering.  
- Knowledge‑graph embeddings (KGE) like TransE/RotatE: map entities/relations to vectors for link prediction, 
similarity, etc.  
- Graph neural networks (GNNs): learn representations that use graph structure and attributes for downstream tasks 
(node classification, relation prediction).  
- Integration with LLMs: KGs can increase factual grounding and reduce hallucination, or be combined with LLMs for 
explainable reasoning.

Tools & ecosystems
- Graph databases: Neo4j (property graph), Amazon Neptune, Blazegraph.  
- Triple stores / RDF: Virtuoso, GraphDB.  
- Open KGs: DBpedia, Wikidata.  
- ML / GNN frameworks: PyTorch Geometric, DGL.  
- Libraries for extraction/indexing: LlamaIndex, spaCy, various IE toolkits.

Typical use cases
- Search and semantic search (better understanding of queries).  
- Question answering and virtual assistants (factual grounding).  
- Recommendations (entity similarity and multi‑hop relations).  
- Data integration and master data management.  
- Scientific knowledge representation, compliance monitoring, digital twins.

Benefits and limitations
- Benefits: explicit, machine‑readable relationships; easier integration across datasets; supports reasoning and 
interpretability.  
- Limitations: curation cost, incompleteness and noise, schema design complexity, scalability and update 
challenges.

Short example
- Triple: (Paris — capitalOf — France)  
- Property graph example: Node[Person: {name: "Alice", birth:1990}] —[:WROTE {year:2021}]-> Node[Paper: 
{title:"..." }]

If you want, I can:
- Show a tiny example KG and some Cypher/SPARQL queries.  
- Walk through building a KG from a small text sample (NER + relation extraction).  
- Explain KG embeddings or GNNs in more detail. Which would you like?

In [50]:
result2 = await agent.run(
    "My name is Filip, I'm a senior data scienstist, specializing in knowledge graphs. What tools and techniques should I know?"
)

In [ ]:
pprint(result2.output)

In [62]:
query_res = database.query(
    "MATCH (n:__memory__)-[r]-(n2) return n.content, r.label, n2"
).result_set
pprint(query_res)

[
    [
        'User: Filip — senior data scientist specializing in knowledge graphs',
        '__mentions__',
        <falkordb.node.Node object at 0x71bdb82b03e0>
    ]
]